# Verify fold 0 — `5s7s10s ~ 2s5s10s` (x7)

Reproduce **exactly** the first walk-forward regression and dump CSVs for Excel.

### What this does
1. Same panel as `run_dz.py backtest` (live + `BACKTEST_START`)
2. Drop NaNs on `(y, x)` → align
3. Train = first **160** clean rows → OLS
4. Predict row **161** → `y_pred`, `residual`
5. Writes:
   - `exports/backtest/verify_train_fold0.csv` ← usa esto en LINEST
   - `exports/backtest/verify_test_fold0.csv`
   - `exports/backtest/verify_summary.csv`

### Excel
1. Abre `verify_train_fold0.csv`
2. `LINEST(y_5s7s10s, x_2s5s10s, TRUE, TRUE)`
3. **Ojo:** LINEST devuelve `[pendiente, intercept]` (pendiente primero)
4. Compara con `verify_summary.csv` / `verify_test_fold0.csv`
5. `y_pred = intercept + beta * x` en la fila test; `residual = y_true - y_pred`

In [ ]:
import importlib, logging, os, sys
from pathlib import Path
import pandas as pd

logging.disable(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: "%.6f" % x)

_DZ = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1")
REPO = Path.cwd() if (Path.cwd() / "lib" / "backtest_verify.py").exists() else _DZ
if not (REPO / "lib" / "backtest_verify.py").exists():
    raise FileNotFoundError("git pull origin main")
os.chdir(REPO)
sys.path = [str(REPO)] + [p for p in sys.path if str(Path(p).resolve()) != str(REPO.resolve())]
print("REPO:", REPO.resolve())

In [ ]:
import run_dz
importlib.reload(run_dz)
run_dz.bootstrap_for_repo(str(REPO))

In [ ]:
import lib.backtest_verify as backtest_verify
importlib.reload(backtest_verify)

out = backtest_verify.verify_fold0(
    target="5s7s10s",
    features=("2s5s10s",),
    # train_size=160,   # default = curve_config.TRAIN_SIZE
    # start / end default = BACKTEST_START / BACKTEST_END
    compare_csv="backtest_x7.csv",
)
print("\nfiles:")
for p in out["paths"]:
    print(" ", p)

In [ ]:
print("=== summary ===")
display(pd.DataFrame([out["summary"]]).T.rename(columns={0: "value"}))
print("\n=== train tail (last 5 of 160) ===")
display(out["train"].tail())
print("\n=== test (row 161 / fold 0 OOS) ===")
display(out["test"])

### Si tu Excel no coincide
- ¿Usaste las filas de **`verify_train_fold0.csv`** (no las primeras 160 del structures crudo)?
- Tras `BACKTEST_START` + dropna el orden cambia.
- LINEST: la **pendiente va primero**, el intercept segundo.
- Misma `time` de test que `verify_test_fold0.csv` / 1ª fila de `backtest_x7.csv`.